In [ ]:
import os
import json
from collections import defaultdict

from pycocotools import mask as maskUtils
import numpy as np
from PIL import Image

# 入力の COCO 形式 JSON のパス
jsonpath = "../../Sandbox/SegLabel_Test/json/result.json"
# 出力マスク画像を保存するディレクトリ
save_dir = "../../Sandbox/SegLabel_Test/mask"
os.makedirs(save_dir, exist_ok=True)

# COCO JSON を読み込み
with open(jsonpath, "r", encoding="utf-8") as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]

# 画像ごとにアノテーションをまとめる
anns_per_image = defaultdict(list)
for ann in annotations:
    anns_per_image[ann["image_id"]].append(ann)

for img_info in images:
    image_id = img_info["id"]
    file_name = img_info["file_name"]
    width = img_info["width"]
    height = img_info["height"]

    # semantic segmentation マスク（背景=0）
    mask = np.zeros((height, width), dtype=np.uint8)

    for ann in anns_per_image.get(image_id, []):
        seg = ann["segmentation"]
        # 画素値として書き込むラベル（ここでは COCO の category_id を使用）
        category_id = ann.get("category_id", 1)

        # segmentation から RLE を生成
        rle = maskUtils.frPyObjects(seg, height, width)
        ann_mask = maskUtils.decode(rle)

        # インスタンスが複数ある場合 (H, W, N) になるので統合
        if ann_mask.ndim == 3:
            ann_mask = np.any(ann_mask, axis=2)
        else:
            ann_mask = ann_mask.astype(bool)

        # 該当画素をカテゴリIDで塗りつぶし（それ以外は背景=0）
        mask[ann_mask] = category_id*255

    # 出力ファイル名は元画像名から拡張子を .png にしたもの
    out_name = os.path.splitext(os.path.basename(file_name))[0][9:] + "_mask.png"
    out_path = os.path.join(save_dir, out_name)

    # 8bit グレースケール画像として保存
    Image.fromarray(mask, mode="L").save(out_path)
    print(f"saved: {out_path}")


saved: ../../Sandbox/SegLabel_Test/mask\b4_HouseCollapse_2.png
saved: ../../Sandbox/SegLabel_Test/mask\b4_HouseCollapse_6.png


C:\Users\kyohe\AppData\Local\Temp\ipykernel_10944\3520788396.py:59: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(mask, mode="L").save(out_path)
